# Attention Mechanism Math — Exercises

8 exercises covering the core mathematics of attention in Transformers.

| # | Exercise | Topics |
|---|----------|--------|
| 1 | Manual Scaled Dot-Product Attention | Q, K, V → scores → scale → softmax → output |
| 2 | √dₖ Scaling Proof (Numerical) | Var(q·k) = dₖ, effect on softmax |
| 3 | Causal Mask Implementation | Masking, sliding window, combined masks |
| 4 | Parameter Counting (LLaMA-3 8B) | MHA, FFN, GQA, full model |
| 5 | KV Cache Memory Calculator | Memory at scale, MHA vs GQA comparison |
| 6 | Attention Entropy Analysis | Entropy computation, interpretation |
| 7 | RoPE by Hand | Rotation matrices, relative position verification |
| 8 | FlashAttention Tiling | Block-tiled attention, online softmax |

Each exercise has: **description → scaffold → solution**.

See [notes.md](notes.md) and [theory.ipynb](theory.ipynb) for reference.

In [ ]:
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['figure.figsize'] = (10, 6)
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not available — text-based outputs only")

def softmax(x, axis=-1):
    """Numerically stable softmax."""
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

---
## Exercise 1: Manual Scaled Dot-Product Attention

For **n=3 tokens** with **dₖ=dᵥ=2**, compute attention step by step:

Given:
```
Q = [[1, 0],     K = [[1, 1],     V = [[1, 0],
     [0, 1],          [0, 1],          [0, 1],
     [1, 1]]          [1, 0]]          [1, 1]]
```

1. Compute raw scores S = QKᵀ (3×3 matrix)
2. Scale: S̃ = S / √dₖ
3. Apply softmax row-wise → attention weights A
4. Compute output O = AV
5. Verify: each row of O is a convex combination of V rows (weights ≥ 0, sum to 1)

In [ ]:
# === Exercise 1: Scaffold ===

Q = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
K = np.array([[1, 1], [0, 1], [1, 0]], dtype=float)
V = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
d_k = 2

# TODO: Step 1 — Compute S = Q @ K.T
S = ...  # shape (3, 3)

# TODO: Step 2 — Scale by √d_k
S_scaled = ...  # shape (3, 3)

# TODO: Step 3 — Softmax (row-wise)
A = ...  # shape (3, 3), each row sums to 1

# TODO: Step 4 — Compute output O = A @ V
O = ...  # shape (3, 2)

# TODO: Step 5 — Verify convex combination
# Check: all weights ≥ 0 and each row sums to 1
# Check: O[i] = sum_j A[i,j] * V[j]

In [ ]:
# === Exercise 1: Solution ===

Q = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
K = np.array([[1, 1], [0, 1], [1, 0]], dtype=float)
V = np.array([[1, 0], [0, 1], [1, 1]], dtype=float)
d_k = 2

# Step 1: Raw scores
S = Q @ K.T
print("Step 1 — S = QKᵀ:")
print(S)
# S[0,0] = [1,0]·[1,1] = 1, S[0,1] = [1,0]·[0,1] = 0, S[0,2] = [1,0]·[1,0] = 1
# S[1,0] = [0,1]·[1,1] = 1, S[1,1] = [0,1]·[0,1] = 1, S[1,2] = [0,1]·[1,0] = 0
# S[2,0] = [1,1]·[1,1] = 2, S[2,1] = [1,1]·[0,1] = 1, S[2,2] = [1,1]·[1,0] = 1

# Step 2: Scale
S_scaled = S / np.sqrt(d_k)
print(f"\nStep 2 — S̃ = S / √{d_k} = S / {np.sqrt(d_k):.4f}:")
print(S_scaled)

# Step 3: Softmax
A = softmax(S_scaled)
print(f"\nStep 3 — A = softmax(S̃):")
print(np.round(A, 4))
print(f"Row sums: {A.sum(axis=1).round(6)}")

# Step 4: Output
O = A @ V
print(f"\nStep 4 — O = AV:")
print(np.round(O, 4))

# Step 5: Verify convex combination
print(f"\nStep 5 — Verification:")
print(f"All weights ≥ 0: {np.all(A >= 0)}")
print(f"Row sums = 1:    {np.allclose(A.sum(axis=1), 1.0)}")
for i in range(3):
    manual = sum(A[i, j] * V[j] for j in range(3))
    print(f"  O[{i}] = {O[i].round(4)} = Σ A[{i},j]·V[j] = {np.array(manual).round(4)} ✓")

---
## Exercise 2: √dₖ Scaling Proof (Numerical)

**Prove numerically** that if $q, k \sim \mathcal{N}(0, I_{d_k})$, then $\text{Var}(q \cdot k) = d_k$.

1. For each $d_k \in \{1, 4, 16, 64, 256, 512\}$:
   - Sample 50,000 random q, k pairs
   - Compute dot products
   - Measure variance (unscaled and scaled by √dₖ)
2. Show that softmax becomes nearly one-hot without scaling for large dₖ
3. Show that scaling restores well-behaved softmax distributions

In [ ]:
# === Exercise 2: Scaffold ===
np.random.seed(42)
n_trials = 50_000
dk_values = [1, 4, 16, 64, 256, 512]

for dk in dk_values:
    q = np.random.randn(n_trials, dk)
    k = np.random.randn(n_trials, dk)
    
    # TODO: Compute dot products
    dots = ...  # shape (n_trials,)
    
    # TODO: Compute variance of unscaled dots
    var_unscaled = ...
    
    # TODO: Compute variance of scaled dots (divided by √dk)
    var_scaled = ...
    
    # TODO: Show softmax behaviour
    # Take 8 random scores and apply softmax with and without scaling
    # max(softmax(random_scores)) → how peaked is the distribution?
    pass

In [ ]:
# === Exercise 2: Solution ===
np.random.seed(42)
n_trials = 50_000
dk_values = [1, 4, 16, 64, 256, 512]

print(f"{'d_k':>6} | {'Var(q·k)':>12} | {'Expected':>10} | {'Var(scaled)':>12} | {'Max softmax (unscaled)':>22} | {'Max softmax (scaled)':>22}")
print("-" * 100)

for dk in dk_values:
    q = np.random.randn(n_trials, dk)
    k = np.random.randn(n_trials, dk)
    
    # Dot products
    dots = np.sum(q * k, axis=1)
    
    # Variances
    var_unscaled = np.var(dots)
    var_scaled = np.var(dots / np.sqrt(dk))
    
    # Softmax peakedness: take 8 "tokens" worth of scores
    scores_8 = np.random.randn(8) * np.sqrt(dk)  # simulate unscaled dot products
    max_unscaled = softmax(scores_8).max()
    max_scaled = softmax(scores_8 / np.sqrt(dk)).max()
    
    print(f"{dk:>6} | {var_unscaled:>12.2f} | {dk:>10} | {var_scaled:>12.4f} | {max_unscaled:>22.6f} | {max_scaled:>22.6f}")

print(f"\n✓ Var(q·k) ≈ d_k for all values")
print(f"✓ Var(q·k / √d_k) ≈ 1.0 for all values")
print(f"✓ Without scaling: softmax becomes one-hot for large d_k")
print(f"✓ With scaling: softmax stays smooth regardless of d_k")

---
## Exercise 3: Causal Mask Implementation

1. Build a **4×4 causal mask** M where M[i,j] = 0 if j ≤ i, else −∞
2. Apply to a random score matrix and verify A[i,j] = 0 for all j > i
3. Build a **sliding window mask** with window size w=2
4. Combine causal + sliding window masks
5. Verify the combined mask allows position 3 to only see positions {1, 2, 3}

In [ ]:
# === Exercise 3: Scaffold ===
np.random.seed(42)
n = 4

# TODO: Build causal mask (4×4)
# M[i,j] = 0 if j <= i, else -inf
M_causal = ...

# TODO: Apply to random scores and verify
scores = np.random.randn(n, n)
# A = softmax(scores + M_causal)
# Check: A[i,j] == 0 for j > i

# TODO: Build sliding window mask (w=2)
# Token i attends to [i-2, i+2]
M_window = ...

# TODO: Combine causal + window
M_combined = ...

# TODO: Verify position 3 can only attend to {1, 2, 3}

In [ ]:
# === Exercise 3: Solution ===
np.random.seed(42)
n = 4

# 1. Causal mask
M_causal = np.zeros((n, n))
M_causal[np.triu_indices(n, k=1)] = -np.inf
print("Causal mask M:")
print(np.where(M_causal == -np.inf, '-inf', '   0'))

# 2. Apply and verify
scores = np.random.randn(n, n)
A = softmax(scores + M_causal)
print(f"\nAttention weights A (with causal mask):")
print(np.round(A, 4))
print(f"A[0,2] (future) = {A[0,2]:.8f} (should be 0)")
print(f"A[0,3] (future) = {A[0,3]:.8f} (should be 0)")
print(f"All future weights zero: {np.allclose(A[np.triu_indices(n, k=1)], 0)}")

# 3. Sliding window mask (w=2)
w = 2
M_window = np.full((n, n), -np.inf)
for i in range(n):
    lo = max(0, i - w)
    hi = min(n, i + w + 1)
    M_window[i, lo:hi] = 0
print(f"\nSliding window mask (w={w}):")
print(np.where(M_window == -np.inf, '-inf', '   0'))

# 4. Combined
M_combined = np.where((M_causal + M_window) < -1e10, -np.inf, 0)
print(f"\nCombined mask (causal + window):")
print(np.where(M_combined == -np.inf, '-inf', '   0'))

# 5. Verify position 3
A_combined = softmax(scores + M_combined)
nonzero_positions = [j for j in range(n) if A_combined[3, j] > 1e-8]
print(f"\nPosition 3 attends to: {nonzero_positions}")
print(f"Expected: [1, 2, 3]  (causal: ≤3, window: ≥1)")
print(f"Correct: {nonzero_positions == [1, 2, 3]}")

---
## Exercise 4: Parameter Counting (LLaMA-3 8B)

For **LLaMA-3 8B** with:
- d_model = 4096, n_heads = 32, n_kv_heads = 8 (GQA), d_head = 128
- d_ff = 14336 (SwiGLU FFN), n_layers = 32
- vocab_size = 128,256

Compute:
1. MHA parameters per layer (Q, K, V, O projections with GQA)
2. FFN parameters per layer (SwiGLU: gate + up + down)
3. Norm parameters per layer
4. Total model parameters
5. Compare to the official 8.03B parameter count

In [ ]:
# === Exercise 4: Scaffold ===

d_model = 4096
n_heads = 32
n_kv_heads = 8
d_head = 128  # = d_model / n_heads
d_ff = 14336
n_layers = 32
vocab_size = 128_256

# TODO: MHA per layer
# Q projection: d_model → n_heads * d_head
q_params = ...
# K projection: d_model → n_kv_heads * d_head  (GQA!)
k_params = ...
# V projection: d_model → n_kv_heads * d_head  (GQA!)
v_params = ...
# O projection: n_heads * d_head → d_model
o_params = ...
# mha_total = q_params + k_params + v_params + o_params

# TODO: FFN per layer (SwiGLU: gate, up, down)
# gate: d_model → d_ff
# up:   d_model → d_ff
# down: d_ff → d_model
ffn_total = ...

# TODO: Total
# embedding + layers * (mha + ffn + norms) + final_norm + lm_head
total = ...

In [ ]:
# === Exercise 4: Solution ===

d_model = 4096
n_heads = 32
n_kv_heads = 8         # GQA with 8 KV groups
d_head = 128            # d_model / n_heads
d_ff = 14336
n_layers = 32
vocab_size = 128_256

# === MHA per layer (with GQA) ===
q_params = d_model * (n_heads * d_head)      # 4096 × 4096 = 16,777,216
k_params = d_model * (n_kv_heads * d_head)   # 4096 × 1024 = 4,194,304  (GQA: 4× fewer than Q)
v_params = d_model * (n_kv_heads * d_head)   # 4096 × 1024 = 4,194,304
o_params = (n_heads * d_head) * d_model      # 4096 × 4096 = 16,777,216
mha_total = q_params + k_params + v_params + o_params

print("=== LLaMA-3 8B Parameter Count ===\n")
print(f"MHA per layer:")
print(f"  W_Q: {d_model} × {n_heads * d_head} = {q_params:>12,}")
print(f"  W_K: {d_model} × {n_kv_heads * d_head} = {k_params:>12,}  (GQA: {n_kv_heads} KV heads, not {n_heads})")
print(f"  W_V: {d_model} × {n_kv_heads * d_head} = {v_params:>12,}")
print(f"  W_O: {n_heads * d_head} × {d_model} = {o_params:>12,}")
print(f"  MHA total:         {mha_total:>12,}")

# === FFN per layer (SwiGLU) ===
gate_params = d_model * d_ff     # 4096 × 14336
up_params = d_model * d_ff       # 4096 × 14336
down_params = d_ff * d_model     # 14336 × 4096
ffn_total = gate_params + up_params + down_params

print(f"\nFFN per layer (SwiGLU):")
print(f"  W_gate: {d_model} × {d_ff} = {gate_params:>12,}")
print(f"  W_up:   {d_model} × {d_ff} = {up_params:>12,}")
print(f"  W_down: {d_ff} × {d_model} = {down_params:>12,}")
print(f"  FFN total:         {ffn_total:>12,}")

# === Norms (RMSNorm: only γ, no bias) ===
norm_params = 2 * d_model  # 2 norms per layer (pre-attn + pre-FFN)

# === Total ===
embedding = vocab_size * d_model
per_layer = mha_total + ffn_total + norm_params
all_layers = per_layer * n_layers
final_norm = d_model
lm_head = vocab_size * d_model  # not tied in LLaMA-3

total = embedding + all_layers + final_norm + lm_head

print(f"\nPer layer: {per_layer:>14,}")
print(f"All {n_layers} layers: {all_layers:>14,}")
print(f"Embedding:     {embedding:>14,}")
print(f"LM head:       {lm_head:>14,}")
print(f"Final norm:    {final_norm:>14,}")
print(f"\n{'='*40}")
print(f"Total:         {total:>14,}  ({total/1e9:.3f}B)")
print(f"Official:      8,030,000,000  (8.030B)")
print(f"Accuracy:      {total/8_030_000_000*100:.1f}%")

---
## Exercise 5: KV Cache Memory Calculator

Compute KV cache memory for **LLaMA-3 70B**:
- d_model = 8192, n_kv_heads = 8, d_head = 128, n_layers = 80

1. KV cache at n = 8,192 tokens (fp16)
2. KV cache at n = 128,000 tokens (fp16)
3. Same at int8 and int4
4. Model weight size (70.6B params, fp16)
5. At what sequence length does KV cache exceed model weights?

In [ ]:
# === Exercise 5: Scaffold ===

d_model = 8192
n_kv_heads = 8
d_head = 128
n_layers = 80
total_params = 70.6e9

# KV cache formula:
# bytes = 2 (K+V) × seq_len × n_kv_heads × d_head × n_layers × bytes_per_elem

# TODO: Compute KV cache at n=8192 (fp16)
kv_8k_fp16 = ...

# TODO: Compute KV cache at n=128000 (fp16)
kv_128k_fp16 = ...

# TODO: Same at int8 and int4

# TODO: Model weight size at fp16
weight_size = ...

# TODO: Find crossover point (when KV cache = model weights)

In [ ]:
# === Exercise 5: Solution ===

d_model = 8192
n_kv_heads = 8
d_head = 128
n_layers = 80
total_params = 70.6e9

def kv_cache_bytes(seq_len, n_kv_heads, d_head, n_layers, bytes_per_elem):
    return 2 * seq_len * n_kv_heads * d_head * n_layers * bytes_per_elem

def to_gb(b):
    return b / (1024**3)

# Model weight size
weight_fp16 = total_params * 2  # fp16 = 2 bytes

print("=== LLaMA-3 70B KV Cache Analysis ===\n")
print(f"Model weights (fp16): {to_gb(weight_fp16):.1f} GB\n")

configs = [
    ("n=8,192, fp16",    8192,    2),
    ("n=8,192, int8",    8192,    1),
    ("n=8,192, int4",    8192,    0.5),
    ("n=128K, fp16",     128000,  2),
    ("n=128K, int8",     128000,  1),
    ("n=128K, int4",     128000,  0.5),
]

print(f"{'Configuration':<20} | {'KV Cache':>10} | {'vs Weights':>12} | {'Status':>20}")
print("-" * 70)

for name, seq_len, bpe in configs:
    kv = kv_cache_bytes(seq_len, n_kv_heads, d_head, n_layers, bpe)
    ratio = kv / weight_fp16
    status = "⚠️ EXCEEDS weights" if kv > weight_fp16 else "OK"
    print(f"{name:<20} | {to_gb(kv):>8.1f}GB | {ratio:>11.2%} | {status:>20}")

# Crossover point
# KV cache = weights when:
# 2 * n * n_kv_heads * d_head * n_layers * 2 = total_params * 2
# n = total_params / (2 * n_kv_heads * d_head * n_layers)
crossover_n = total_params / (2 * n_kv_heads * d_head * n_layers)
print(f"\nCrossover: KV cache (fp16) = model weights at n = {crossover_n:,.0f} tokens")
print(f"Beyond this, KV cache dominates total memory.")

# With MHA (all 64 heads as KV heads) for comparison
n_kv_mha = 64  # if no GQA
crossover_mha = total_params / (2 * n_kv_mha * d_head * n_layers)
print(f"\nWith standard MHA (64 KV heads): crossover at n = {crossover_mha:,.0f} tokens")
print(f"GQA extends the crossover by {crossover_n / crossover_mha:.0f}× (= H/G = 64/8 = 8×)")

---
## Exercise 6: Attention Entropy Analysis

Compute entropy $H(A_i) = -\sum_j A_{ij} \log A_{ij}$ for:

1. **Uniform**: A = [0.25, 0.25, 0.25, 0.25] → H = log(4)
2. **Peaked**: A = [0.97, 0.01, 0.01, 0.01] → low entropy
3. **One-hot**: A = [1, 0, 0, 0] → H = 0

Then: interpret what each pattern means for information flow through the attention layer.

In [ ]:
# === Exercise 6: Scaffold ===

# TODO: Compute entropy for each distribution
# H(A) = -sum(A * log(A))  (handle 0·log(0) = 0)

A_uniform = np.array([0.25, 0.25, 0.25, 0.25])
A_peaked = np.array([0.97, 0.01, 0.01, 0.01])
A_onehot = np.array([1.0, 0.0, 0.0, 0.0])

# For each distribution:
# 1. Compute H
# 2. Compare to max entropy (log n)
# 3. Interpret: what does this entropy mean for the attention output?

In [ ]:
# === Exercise 6: Solution ===

def entropy(p):
    """Compute entropy H(p) = -sum(p * log(p)), handling 0s correctly."""
    p_safe = p[p > 0]  # ignore zero entries: 0·log(0) = 0
    return -np.sum(p_safe * np.log(p_safe))

n = 4
max_entropy = np.log(n)

distributions = {
    "Uniform [0.25, 0.25, 0.25, 0.25]": np.array([0.25, 0.25, 0.25, 0.25]),
    "Peaked  [0.97, 0.01, 0.01, 0.01]": np.array([0.97, 0.01, 0.01, 0.01]),
    "One-hot [1.00, 0.00, 0.00, 0.00]": np.array([1.0, 0.0, 0.0, 0.0]),
    "Moderate [0.4, 0.3, 0.2, 0.1]":    np.array([0.4, 0.3, 0.2, 0.1]),
}

print(f"Max entropy H_max = log({n}) = {max_entropy:.4f}\n")
print(f"{'Distribution':<40} | {'H':>8} | {'H/H_max':>8} | {'Interpretation'}")
print("-" * 95)

interpretations = {
    "Uniform": "Equal information from all tokens → broadcasting/averaging",
    "Peaked":  "Almost all info from one token → specific dependency (e.g., subject-verb)",
    "One-hot": "Exactly one source → hard copy operation",
    "Moderate":"Graded importance → soft retrieval with preferences",
}

for name, p in distributions.items():
    H = entropy(p)
    H_norm = H / max_entropy
    key = name.split()[0]
    interp = interpretations.get(key, "")
    print(f"{name:<40} | {H:>8.4f} | {H_norm:>7.2%} | {interp}")

# Visualise with value vectors
print("\n=== Effect on Output ===")
V = np.array([[1, 0], [0, 1], [-1, 0], [0, -1]], dtype=float)
print(f"V (4 value vectors, 2D): {V.tolist()}")
for name, p in distributions.items():
    output = p @ V
    print(f"  {name.split('[')[0].strip()}: output = {output.round(4)} (magnitude = {np.linalg.norm(output):.4f})")

print("\nUniform → output ≈ mean(V) → washes out signal")
print("One-hot → output = exact copy of one V → preserves specific information")
print("Real attention heads span this spectrum; both extremes serve different functions.")

---
## Exercise 7: RoPE by Hand

For **d=4, position m=2**, with frequencies θ = (1.0, 0.01):

1. Build the 4×4 block-diagonal rotation matrix $R_2$
2. Apply to q = (1, 0, 1, 0)
3. Apply to k = (0, 1, 0, 1) at positions n=0,1,2,3
4. Verify: dot(R_m q, R_n k) depends only on (m − n), not on m or n individually

In [ ]:
# === Exercise 7: Scaffold ===

d = 4
m = 2
thetas = np.array([1.0, 0.01])  # two 2D rotation blocks
q = np.array([1.0, 0.0, 1.0, 0.0])
k = np.array([0.0, 1.0, 0.0, 1.0])

# TODO: Step 1 — Build R_2 (4×4 block-diagonal rotation matrix)
# Block 0: rotate by m * θ_0 = 2 * 1.0 = 2 radians
# Block 1: rotate by m * θ_1 = 2 * 0.01 = 0.02 radians
R_2 = ...  # (4, 4)

# TODO: Step 2 — Apply R_2 to q
q_rotated = ...

# TODO: Step 3 — Apply R_n to k at positions n=0,1,2,3

# TODO: Step 4 — Verify dot(R_m q, R_n k) = f(m-n)

In [ ]:
# === Exercise 7: Solution ===

d = 4
thetas = np.array([1.0, 0.01])
q = np.array([1.0, 0.0, 1.0, 0.0])
k = np.array([0.0, 1.0, 0.0, 1.0])

def build_rope_matrix(pos, thetas):
    """Build block-diagonal rotation matrix R_pos."""
    d = len(thetas) * 2
    R = np.eye(d)
    for i, theta in enumerate(thetas):
        angle = pos * theta
        c, s = np.cos(angle), np.sin(angle)
        idx = 2 * i
        R[idx, idx] = c
        R[idx, idx+1] = -s
        R[idx+1, idx] = s
        R[idx+1, idx+1] = c
    return R

def apply_rotation(vec, pos, thetas):
    return build_rope_matrix(pos, thetas) @ vec

# Step 1: Build R_2
m = 2
R_2 = build_rope_matrix(m, thetas)
print("Step 1 — R_2 (rotation matrix at position 2):")
print(np.round(R_2, 6))

# Step 2: Apply to q
q_rot = R_2 @ q
print(f"\nStep 2 — R_2 · q = {q_rot.round(6)}")
print(f"  Block 0: rotate (1, 0) by 2·1.0 = 2 rad → ({np.cos(2):.4f}, {np.sin(2):.4f})")
print(f"  Block 1: rotate (1, 0) by 2·0.01 = 0.02 rad → ({np.cos(0.02):.4f}, {np.sin(0.02):.4f})")

# Step 3: Apply to k at different positions
print(f"\nStep 3 — R_n · k for various n:")
for n in range(5):
    k_rot = apply_rotation(k, n, thetas)
    print(f"  n={n}: R_{n}·k = {k_rot.round(6)}")

# Step 4: Verify relative position property
print(f"\nStep 4 — dot(R_m q, R_n k) for various (m, n):")
print(f"{'m':>4} {'n':>4} {'m-n':>5} {'dot product':>14}")
print("-" * 30)

# Collect dots by offset
offset_dots = {}
for m in range(6):
    for n in range(6):
        q_r = apply_rotation(q, m, thetas)
        k_r = apply_rotation(k, n, thetas)
        dot_val = np.dot(q_r, k_r)
        offset = m - n
        if offset not in offset_dots:
            offset_dots[offset] = []
        offset_dots[offset].append(dot_val)
        if abs(m - n) <= 2 and m < 4 and n < 4:
            print(f"{m:>4} {n:>4} {m-n:>5} {dot_val:>14.8f}")

print(f"\nVerification: same offset → same dot product:")
for offset in sorted(offset_dots.keys()):
    vals = offset_dots[offset]
    all_equal = np.allclose(vals, vals[0], atol=1e-10)
    print(f"  offset {offset:>3}: all equal = {all_equal} (value = {vals[0]:.8f})")

print(f"\n✓ RoPE makes attention scores depend ONLY on relative position (m−n)")

## Exercise 8: FlashAttention — Block-Tiled Attention with Online Softmax

**Difficulty**: ★★★★ (Hard)

**Goal**: Implement the FlashAttention tiling algorithm from scratch.

Standard attention materializes the full $n \times n$ score matrix, requiring $O(n^2)$ memory.
FlashAttention processes attention in **tiles** using the **online softmax trick**:

Instead of computing softmax over the full row at once, we process blocks of keys/values and
**incrementally update** the running softmax numerator and denominator.

**The online softmax update equations:**

Given a running maximum $m_{old}$, unnormalized accumulator $\ell_{old}$, and output $O_{old}$,
when processing a new block with scores $S_{block}$:

$$m_{new} = \max(m_{old}, \max(S_{block}))$$
$$\ell_{new} = e^{m_{old} - m_{new}} \cdot \ell_{old} + \sum_j e^{S_{block,j} - m_{new}}$$
$$O_{new} = \frac{e^{m_{old} - m_{new}} \cdot \ell_{old} \cdot O_{old} + \sum_j e^{S_{block,j} - m_{new}} \cdot V_{block,j}}{\ell_{new}}$$

**Tasks:**
1. Implement `standard_attention(Q, K, V)` as a reference
2. Implement `tiled_attention(Q, K, V, block_size)` using online softmax
3. Verify outputs match within numerical tolerance
4. Compare peak memory (count of floats simultaneously alive)
5. Test with `n=16, d=8, block_size=4`

In [ ]:
# === Exercise 8: Scaffold ===

np.random.seed(42)
n, d = 16, 8
block_size = 4

Q = np.random.randn(n, d) * 0.5
K = np.random.randn(n, d) * 0.5
V = np.random.randn(n, d) * 0.5

def softmax(x, axis=-1):
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)

def standard_attention(Q, K, V):
    """Standard scaled dot-product attention (full materialization)."""
    d_k = Q.shape[-1]
    # TODO: Compute scores, scale, softmax, output
    # Memory: O(n^2) for the full score matrix
    pass

def tiled_attention(Q, K, V, block_size):
    """
    FlashAttention-style tiled attention using online softmax.
    
    Process Q in blocks of block_size rows.
    For each Q block, iterate over K/V blocks and incrementally
    update the running softmax using the online trick.
    """
    n, d = Q.shape
    d_k = d
    scale = 1.0 / np.sqrt(d_k)
    O = np.zeros((n, d))
    
    num_q_blocks = (n + block_size - 1) // block_size
    
    for q_block_idx in range(num_q_blocks):
        q_start = q_block_idx * block_size
        q_end = min(q_start + block_size, n)
        Q_block = Q[q_start:q_end]  # (block_q, d)
        bq = q_end - q_start
        
        # Running accumulators for online softmax
        m_running = np.full(bq, -np.inf)        # running row-wise max
        ell_running = np.zeros(bq)                # running sum of exp
        O_running = np.zeros((bq, d))             # running weighted output
        
        num_kv_blocks = (n + block_size - 1) // block_size
        
        for kv_block_idx in range(num_kv_blocks):
            kv_start = kv_block_idx * block_size
            kv_end = min(kv_start + block_size, n)
            K_block = K[kv_start:kv_end]  # (block_kv, d)
            V_block = V[kv_start:kv_end]
            
            # TODO: Compute block scores S_block = Q_block @ K_block^T * scale
            # TODO: Find new row-wise max m_new
            # TODO: Rescale old accumulators
            # TODO: Compute new exp weights
            # TODO: Update ell_running and O_running
            pass
        
        # TODO: Normalize O_running by ell_running
        # TODO: Store result in O[q_start:q_end]
        pass
    
    return O

# Test
O_std = standard_attention(Q, K, V)
O_tiled = tiled_attention(Q, K, V, block_size)

if O_std is not None and O_tiled is not None:
    max_err = np.max(np.abs(O_std - O_tiled))
    print(f"Max error: {max_err:.2e}")
    print(f"Match: {np.allclose(O_std, O_tiled, atol=1e-6)}")
else:
    print("TODO: Implement both functions")

In [ ]:
# === Exercise 8: Solution ===

np.random.seed(42)
n, d = 16, 8
block_size = 4

Q = np.random.randn(n, d) * 0.5
K = np.random.randn(n, d) * 0.5
V = np.random.randn(n, d) * 0.5

def softmax_rows(x):
    e = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return e / np.sum(e, axis=-1, keepdims=True)

# --- Part 1: Standard attention ---
def standard_attention(Q, K, V):
    d_k = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(d_k)   # (n, n) — full materialization
    weights = softmax_rows(scores)
    return weights @ V

# --- Part 2: Tiled attention with online softmax ---
def tiled_attention(Q, K, V, block_size):
    n, d = Q.shape
    scale = 1.0 / np.sqrt(d)
    O = np.zeros((n, d))
    
    num_q_blocks = (n + block_size - 1) // block_size
    
    for q_idx in range(num_q_blocks):
        q_s = q_idx * block_size
        q_e = min(q_s + block_size, n)
        Q_b = Q[q_s:q_e]          # (bq, d)
        bq = q_e - q_s
        
        # Online softmax accumulators (per row of Q_b)
        m_i = np.full(bq, -np.inf)     # running max
        ell_i = np.zeros(bq)            # running sum of exp
        O_i = np.zeros((bq, d))         # running unnormalized output
        
        num_kv_blocks = (n + block_size - 1) // block_size
        
        for kv_idx in range(num_kv_blocks):
            kv_s = kv_idx * block_size
            kv_e = min(kv_s + block_size, n)
            K_b = K[kv_s:kv_e]    # (bkv, d)
            V_b = V[kv_s:kv_e]    # (bkv, d)
            
            # Block scores
            S_b = Q_b @ K_b.T * scale  # (bq, bkv)
            
            # New maximum per row
            m_block = np.max(S_b, axis=-1)        # (bq,)
            m_new = np.maximum(m_i, m_block)       # (bq,)
            
            # Correction factor for old accumulators
            correction = np.exp(m_i - m_new)       # (bq,)
            
            # Exp weights for this block
            P_b = np.exp(S_b - m_new[:, None])     # (bq, bkv)
            
            # Update running sum
            ell_new = correction * ell_i + np.sum(P_b, axis=-1)  # (bq,)
            
            # Update running output (keep unnormalized)
            O_i = correction[:, None] * O_i + P_b @ V_b  # (bq, d)
            
            # Update state
            m_i = m_new
            ell_i = ell_new
        
        # Normalize
        O[q_s:q_e] = O_i / ell_i[:, None]
    
    return O

# --- Part 3: Verify correctness ---
O_std = standard_attention(Q, K, V)
O_tiled = tiled_attention(Q, K, V, block_size)

max_err = np.max(np.abs(O_std - O_tiled))
mean_err = np.mean(np.abs(O_std - O_tiled))
print("=== FlashAttention Tiling Verification ===")
print(f"n={n}, d={d}, block_size={block_size}")
print(f"Max absolute error:  {max_err:.2e}")
print(f"Mean absolute error: {mean_err:.2e}")
print(f"Outputs match: {np.allclose(O_std, O_tiled, atol=1e-10)}")

# --- Part 4: Memory comparison ---
print(f"\n=== Memory Analysis (floats simultaneously alive) ===")

# Standard: materializes full n×n score matrix + n×n weight matrix
std_scores = n * n       # S = Q @ K^T
std_weights = n * n      # softmax(S)
std_peak = std_scores + std_weights + n * d  # + output
print(f"Standard attention:")
print(f"  Score matrix:  {n}×{n} = {std_scores} floats")
print(f"  Weight matrix: {n}×{n} = {std_weights} floats")
print(f"  Peak memory:   {std_peak} floats")

# Tiled: only block_size × block_size alive at once (+ accumulators)
bq = block_size
bkv = block_size
tiled_scores = bq * bkv           # S_block
tiled_exp = bq * bkv              # P_block  
tiled_accum = bq + bq + bq * d    # m_i + ell_i + O_i
tiled_peak = tiled_scores + tiled_exp + tiled_accum
print(f"\nTiled attention (block_size={block_size}):")
print(f"  Block scores:  {bq}×{bkv} = {tiled_scores} floats")
print(f"  Block exp:     {bq}×{bkv} = {tiled_exp} floats")
print(f"  Accumulators:  m({bq}) + ℓ({bq}) + O({bq}×{d}) = {tiled_accum} floats")
print(f"  Peak memory:   {tiled_peak} floats")
print(f"\nMemory reduction: {std_peak/tiled_peak:.1f}×")

# Part 5: Test with different block sizes
print(f"\n=== Block Size Sensitivity ===")
print(f"{'block_size':>10} {'max_error':>12} {'correct':>8}")
print("-" * 32)
for bs in [1, 2, 4, 8, 16]:
    O_t = tiled_attention(Q, K, V, bs)
    err = np.max(np.abs(O_std - O_t))
    ok = np.allclose(O_std, O_t, atol=1e-10)
    print(f"{bs:>10} {err:>12.2e} {'✓' if ok else '✗':>8}")

print(f"\n✓ Online softmax produces exact results regardless of block size")